# 8개 선택 특성 모델 A 비교

원본 최종 수정 기록: 2025-06-01T21:57:12+09:00.

파일명의 번호는 파이프라인 순서이며 버전 번호가 아닙니다. 데이터·모델 경로와 실행 환경을 설정한 뒤 필요한 단계를 실행하세요.


In [ ]:
import pandas as pd

In [ ]:
df_insta = pd.read_csv('comb_Z_score.csv')

In [ ]:
df_x = pd.read_csv('x_Z_score.csv')

In [ ]:
selected_features = [
    "description length",
    "#posts",
    "nums/length username",
    "#followers",
    "#follows",
    "fullname words",
    "private",
    "nums/length fullname"
] #x_data_set에서 p-value를 검증을 했을 때 결과와 유의미하다고 판단된 특성들 만을 활용해서 학습

In [ ]:
df_insta_selected = df_insta[selected_features + ['fake']]

In [ ]:
df_x_selected = df_x[selected_features + ['fake']]

In [ ]:
df_insta_selected

In [ ]:
df_x_selected

In [ ]:
from sklearn.ensemble import VotingClassifier
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
import lightgbm as lgb
import xgboost as xgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import pandas as pd

X = df_insta_selected.drop(columns=["fake"])
y = df_insta_selected["fake"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 개별 모델 정의
model1 = GradientBoostingClassifier(random_state=42)
model2 = RandomForestClassifier(random_state=42)
model3 = lgb.LGBMClassifier()
model4 = xgb.XGBClassifier(use_label_encoder=False, eval_metric='logloss')

# 소프트 보팅 앙상블
voting_clf = VotingClassifier(
    estimators=[
        ('gb', model1),
        ('rf', model2),
        ('lgbm', model3),
        ('xgb', model4)
    ],
    weights = [2,4,2,4],
    voting='soft'
)

# 학습 및 평가
voting_clf.fit(X_train, y_train)
preds = voting_clf.predict(X_test)

# 결과 출력
print("Voting Classifier Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds, target_names=["Real", "Fake"]))

test_preds = voting_clf.predict(vali_X)

print("Voting Classifier Accuracy:", accuracy_score(vali_y, test_preds))
print(classification_report(vali_y, test_preds, target_names=["Real", "Fake"]))


In [ ]:
vali_X = df_x_selected.drop(columns=["fake"])
vali_y = df_x_selected["fake"]

In [ ]:
voting_clf.score(vali_X,vali_y)

In [ ]:
from sklearn.model_selection import train_test_split

X_test_vali, _, y_test_vali, _ = train_test_split(
    vali_X, vali_y,
    test_size=0.8,      
    random_state=42,
    stratify=vali_y    
)


In [ ]:
X_test_vali

In [ ]:
preds = voting_clf.predict(X_test_vali)

# 결과 출력
print("Voting Classifier Accuracy:", accuracy_score(y_test_vali, preds))
print(classification_report(y_test_vali, preds, target_names=["0", "1"]))

In [ ]:
voting_clf.score(X_test_vali,y_test_vali)

In [ ]:
rf_model = RandomForestClassifier(random_state=42)
gb_model = GradientBoostingClassifier(random_state=42)

# 모델 학습
rf_model.fit(X_train, y_train)
gb_model.fit(X_train, y_train)

# 예측
rf_pred = rf_model.predict(X_test_vali)
gb_pred = gb_model.predict(X_test_vali)


In [ ]:
rf_pred

In [ ]:
# 평가 출력
print("✅ Random Forest 결과")
print(classification_report(y_test_vali, rf_pred))
print("정확도:", accuracy_score(y_test_vali, rf_pred))

print("\n✅ Gradient Boosting 결과")
print(classification_report(y_test_vali, gb_pred))
print("정확도:", accuracy_score(y_test_vali, gb_pred))